### Cell 1 -  Xem tất cả experiments

In [0]:
# ============================================================
#  DAY 11 — MLflow Tracking Review
#  Kiểm tra toàn bộ experiments đã log trong project
# ============================================================

import mlflow
import pandas as pd

spark.sql("USE tourgo")

# Tìm experiment
try:
    exp = mlflow.get_experiment_by_name(
        "/Shared/tourgo-ml-experiments"
    )
    if exp is None:
        exp = mlflow.get_experiment_by_name(
            "tourgo-ml-experiments"
        )
    print(f"--OK-- Experiment: {exp.name}")
    print(f"   ID: {exp.experiment_id}")
except Exception as e:
    print(f"--ERROR {e}")

--OK-- Experiment: /Shared/tourgo-ml-experiments
   ID: 2112271813259096


### Cell 2 — Xem tất cả runs

In [0]:
# Lấy toàn bộ runs
runs = mlflow.search_runs(
    experiment_names=[
        "/Shared/tourgo-ml-experiments",
        "tourgo-ml-experiments"
    ]
)

print("=" * 70)
print("--> MLFLOW RUNS SUMMARY — GỬI CHO [A] HÀ")
print("=" * 70)

# Các cột quan trọng
cols_to_show = [
    "tags.mlflow.runName",
    "status",
    "metrics.final_silhouette",
    "metrics.silhouette_k2",
    "metrics.silhouette_k3",
    "metrics.silhouette_k4",
    "metrics.rmse",
    "metrics.r2",
    "start_time"
]

available_cols = [c for c in cols_to_show if c in runs.columns]
print(runs[available_cols].to_string(index=False))

print("\n" + "=" * 70)
print(f"Tổng số runs: {len(runs)}")

2026/06/04 11:56:50 WARNING mlflow.tracking.fluent: Cannot retrieve experiment by name tourgo-ml-experiments


--> MLFLOW RUNS SUMMARY — GỬI CHO [A] HÀ
   tags.mlflow.runName   status  metrics.final_silhouette  metrics.silhouette_k2  metrics.silhouette_k3  metrics.silhouette_k4  metrics.rmse  metrics.r2                       start_time
revenue_forecasting_lr FINISHED                       NaN                    NaN                    NaN                    NaN  4516569.6586    0.878957 2026-06-02 13:51:07.351000+00:00
       kmeans_final_k4 FINISHED                  0.410941                    NaN                    NaN                    NaN           NaN         NaN 2026-06-02 11:24:18.167000+00:00
   kmeans_elbow_method FINISHED                       NaN               0.530103               0.431731               0.410941           NaN         NaN 2026-06-02 11:21:55.940000+00:00
       kmeans_final_k4   FAILED                       NaN                    NaN                    NaN                    NaN           NaN         NaN 2026-06-02 08:58:41.600000+00:00
   kmeans_elbow_method   FAIL

### Cell 3 — So sánh K-Means models

In [0]:
# ── So sánh các K-Means runs ────────────────────────────────
kmeans_runs = runs[
    runs["tags.mlflow.runName"].str.contains(
        "kmeans", case=False, na=False
    )
].copy()

print("--> K-MEANS MODEL COMPARISON:")
print("=" * 55)

if len(kmeans_runs) > 0:
    sil_cols = [f"metrics.silhouette_k{k}" for k in range(2, 9)
                if f"metrics.silhouette_k{k}" in kmeans_runs.columns]

    for _, row in kmeans_runs.iterrows():
        name = row.get("tags.mlflow.runName", "unknown")
        print(f"\n  Run: {name}")
        for col_name in sil_cols:
            k = col_name.split("_k")[-1]
            val = row.get(col_name)
            if pd.notna(val):
                bar = "█" * int(val * 20)
                print(f"    K={k}: {val:.4f} {bar}")
else:
    print("  --WARNING--  Không tìm thấy K-Means runs")

# Linear Regression run
lr_runs = runs[
    runs["tags.mlflow.runName"].str.contains(
        "forecast|linear|lr", case=False, na=False
    )
]

print("\n--> LINEAR REGRESSION:")
print("=" * 55)
if len(lr_runs) > 0:
    for _, row in lr_runs.iterrows():
        rmse = row.get("metrics.rmse", "N/A")
        r2   = row.get("metrics.r2", "N/A")
        print(f"  RMSE: {rmse:,.0f} VNĐ" if isinstance(rmse, float)
              else f"  RMSE: {rmse}")
        print(f"  R²  : {r2:.4f}" if isinstance(r2, float)
              else f"  R²  : {r2}")

--> K-MEANS MODEL COMPARISON:

  Run: kmeans_final_k4

  Run: kmeans_elbow_method
    K=2: 0.5301 ██████████
    K=3: 0.4317 ████████
    K=4: 0.4109 ████████
    K=5: 0.5667 ███████████
    K=6: 0.4701 █████████
    K=7: 0.5241 ██████████
    K=8: 0.5192 ██████████

  Run: kmeans_final_k4

  Run: kmeans_elbow_method

  Run: kmeans_elbow_method

  Run: kmeans_elbow_method

  Run: kmeans_elbow_method

  Run: kmeans_elbow_method

  Run: kmeans_elbow_method

  Run: kmeans_elbow_method

  Run: kmeans_elbow_method

  Run: kmeans_final_k4

  Run: kmeans_elbow_method
    K=2: 0.5301 ██████████
    K=3: 0.4317 ████████
    K=4: 0.4109 ████████
    K=5: 0.5667 ███████████
    K=6: 0.4701 █████████
    K=7: 0.5241 ██████████
    K=8: 0.5192 ██████████

  Run: kmeans_final_k4

  Run: kmeans_elbow_method
    K=2: 0.5301 ██████████
    K=3: 0.4317 ████████
    K=4: 0.4109 ████████
    K=5: 0.5667 ███████████
    K=6: 0.4701 █████████
    K=7: 0.5241 ██████████

  Run: kmeans_final_k4

  Run: kmeans

### Cell 4 — Checklist Day 10 & 11